# Descriptor 4 : Z + D + Eigenvalues(A) + Eigenvalues(C)
**Molecular Property Prediction — Heat of Vaporization**

Models: Random Forest · XGBoost · LightGBM

Evaluation: **5-fold cross-validation** (shuffle, random_state=42); RF 300 trees, LightGBM 400, XGBoost 400. Reported as mean ± SD over folds: R² and MSE.

In [ ]:
# Install required libraries (run once)
!pip install -q rdkit scikit-learn xgboost lightgbm openpyxl

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
print("All libraries loaded successfully!")

In [ ]:
# Upload your dataset file (Colab) — or read it from the current folder when run locally
try:
    from google.colab import files
    uploaded = files.upload()   # select  Aliphatic_HV_with_SMILES.xlsx
except ImportError:
    # Running locally: look for the dataset next to the notebook or in ../data/
    import os
    for path in ['Aliphatic_HV_with_SMILES.xlsx', os.path.join('..', 'data', 'Aliphatic_HV_with_SMILES.xlsx')]:
        if os.path.exists(path):
            uploaded = {os.path.basename(path): open(path, 'rb').read()}
            break
    else:
        raise FileNotFoundError('Aliphatic_HV_with_SMILES.xlsx not found (expected in this folder or in ../data/)')

In [ ]:
import io
filename = list(uploaded.keys())[0]
df = pd.read_excel(io.BytesIO(uploaded[filename]))
print(f"Dataset Shape : {df.shape}")
print(f"Columns       : {list(df.columns)}")
df.head(10)

In [ ]:
# ── SMILES Parser (custom, no rdkit needed) ─────────────────────────────
ATOMIC_NUM = {'C': 6, 'H': 1}
VALENCE    = {'C': 4}

def parse_smiles(smi):
    atoms, bonds = [], []
    bo_map = {'-':1, '=':2, '#':3}
    stack, rings = [], {}
    i, prev, pbo = 0, None, None
    while i < len(smi):
        ch = smi[i]
        if   ch == '(':  stack.append(prev); i += 1
        elif ch == ')':  prev = stack.pop();  i += 1
        elif ch in '-=#': pbo = bo_map[ch]; i += 1
        elif ch == 'C':
            ai = len(atoms); atoms.append('C')
            bo = pbo or 1
            if prev is not None: bonds.append((prev, ai, bo))
            pbo = None; prev = ai; i += 1
        elif ch == '[':
            j = smi.index(']', i)
            sym = ''.join(c for c in smi[i+1:j] if c.isalpha())
            if sym in ATOMIC_NUM:
                ai = len(atoms); atoms.append(sym)
                bo = pbo or 1
                if prev is not None: bonds.append((prev, ai, bo))
                pbo = None; prev = ai
            i = j + 1
        elif ch.isdigit() or ch == '%':
            if ch == '%': rn = int(smi[i+1:i+3]); i += 3
            else:         rn = int(ch);            i += 1
            bo = pbo or 1; pbo = None
            if rn in rings:
                oa, ob = rings.pop(rn)
                bonds.append((oa, prev, max(bo, ob) if (bo>1 or ob>1) else 1))
            else: rings[rn] = (prev, bo)
        else: i += 1
    return atoms, bonds

def add_H(atoms, bonds):
    n = len(atoms)
    bc = [0]*n
    for a,b,bo in bonds: bc[a]+=bo; bc[b]+=bo
    na, nb2 = list(atoms), list(bonds)
    for i in range(n):
        if atoms[i] == 'C':
            for _ in range(max(0, VALENCE['C'] - bc[i])):
                hi = len(na); na.append('H'); nb2.append((i, hi, 1))
    return na, nb2

def Z_vec(atoms):  return np.array([ATOMIC_NUM[a] for a in atoms], dtype=float)
def Amat(n, bonds):
    A = np.zeros((n,n))
    for a,b,_ in bonds: A[a,b]=A[b,a]=1
    return A
def Dmat(A):   return np.diag(A.sum(1))
def Cmat(atoms, n):
    Z = Z_vec(atoms)
    C = np.zeros((n,n))
    for i in range(n):
        C[i,i] = 0.5*Z[i]**2.4
        for j in range(i+1,n):
            C[i,j] = C[j,i] = Z[i]*Z[j]/abs(i-j)
    return C
def WAmat(n, bonds):
    W = np.zeros((n,n))
    for a,b,bo in bonds: W[a,b]=W[b,a]=bo
    return W

print("SMILES parser ready!")

In [ ]:
# ── Process every molecule (add explicit H) ─────────────────────────────
mol_data = []
MAX_ATOMS = 32   # largest molecule in dataset = 32 atoms (incl. H)

for _, row in df.iterrows():
    at, bo = parse_smiles(row['SMILES'])
    ah, bh = add_H(at, bo)
    mol_data.append({'atoms':ah,'bonds':bh,'n':len(ah)})

sizes = [m['n'] for m in mol_data]
print(f"MAX_ATOMS (including H) : {MAX_ATOMS}")
print(f"Molecule sizes — min:{min(sizes)}  max:{max(sizes)}  mean:{np.mean(sizes):.1f}")

In [ ]:
# ── Descriptor 4 : Z + D + Eigenvalues(A) + Eigenvalues(C) ──────────────
N = MAX_ATOMS
col_names  = [f'Z_{i+1}'    for i in range(N)]
col_names += [f'D_{i+1}'    for i in range(N)]
col_names += [f'EigA_{i+1}' for i in range(N)]
col_names += [f'EigC_{i+1}' for i in range(N)]

rows = []
for m in mol_data:
    n, ah, bh = m['n'], m['atoms'], m['bonds']
    A = Amat(n,bh); D = Dmat(A); C = Cmat(ah,n); Z = Z_vec(ah)
    Zp=np.zeros(N); Zp[:n]=Z
    Dp=np.zeros(N); Dp[:n]=np.diag(D)
    # eigenvalues sorted descending
    eA = np.sort(np.linalg.eigvalsh(A))[::-1]
    eC = np.sort(np.linalg.eigvalsh(C))[::-1]
    eAp=np.zeros(N); eAp[:n]=eA
    eCp=np.zeros(N); eCp[:n]=eC
    rows.append(np.concatenate([Zp, Dp, eAp, eCp]))

X = np.array(rows)
y = df['Heat_of_Vaporization'].values
print(f"Feature matrix : {X.shape}  (Z+D+Eig(A)+Eig(C), 32 eigenvalues each)")

In [ ]:
# ── Show complete feature DataFrame ──────────────────────────────────────
df_feat = pd.DataFrame(X, columns=col_names)
df_feat.insert(0,'Molecule_Name', df['Molecule Name'].values)
df_feat['Heat_of_Vaporization'] = y
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)
print(f"DataFrame shape : {df_feat.shape}")
print(f"Columns         : {len(df_feat.columns)}")
print("\nAll column names:")
print(list(df_feat.columns))
df_feat.head()

In [ ]:
DESC_LABEL = 'Z + D + Eig(A) + Eig(C)'
DESC_TAG   = 'Desc4'

# ── 5-fold cross-validation setup ───────────────────────────────────────
# Same folds (shuffle, random_state=42) and same model settings are used for
# every descriptor, the ECFP4/MACCS benchmarks and the fusion descriptors.
kf = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(kf.split(X))
for k, (tr, te) in enumerate(folds, 1):
    print(f"Fold {k}: train {len(tr)}  |  test {len(te)}")

def make_models():
    return {
        'Random Forest': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
        'LightGBM'     : lgb.LGBMRegressor(n_estimators=400, random_state=42, n_jobs=-1, verbose=-1),
        'XGBoost'      : xgb.XGBRegressor(n_estimators=400, random_state=42, n_jobs=-1, verbosity=0),
    }

In [ ]:
# ── Run 5-fold CV for RF, LightGBM and XGBoost ──────────────────────────
fold_rows, summary_rows = [], []
oof_pred = {}
for name, proto in make_models().items():
    oof = np.zeros_like(y, dtype=float)
    for k, (tr, te) in enumerate(folds, 1):
        model = type(proto)(**proto.get_params())
        model.fit(X[tr], y[tr])
        p = model.predict(X[te]); oof[te] = p
        mse = mean_squared_error(y[te], p)
        fold_rows.append({'Model': name, 'Fold': k, 'R2': r2_score(y[te], p), 'MSE': mse})
    oof_pred[name] = oof
    f = pd.DataFrame([r for r in fold_rows if r['Model'] == name])
    summary_rows.append({'Model': name,
        'R2 (mean)': f.R2.mean(),   'R2 (SD)': f.R2.std(),
        'MSE (mean)': f.MSE.mean(), 'MSE (SD)': f.MSE.std(),
        'R2 (pooled out-of-fold)': r2_score(y, oof)})
    print(f"{name:14s} R² = {f.R2.mean():.4f} ± {f.R2.std():.4f}   MSE = {f.MSE.mean():.3f} ± {f.MSE.std():.3f}")

fold_results = pd.DataFrame(fold_rows)
fold_results

In [ ]:
# ── Final Comparison (mean ± SD over the 5 folds) ───────────────────────
results = pd.DataFrame(summary_rows).sort_values('R2 (mean)', ascending=False).reset_index(drop=True)
print("="*72)
print(f"  DESCRIPTOR: {DESC_LABEL}   —   5-fold cross-validation")
print("="*72)
print(results.round(4).to_string(index=False))
print("="*72)
best = results.iloc[0]
print(f"\n  Best Model : {best['Model']}  (R² = {best['R2 (mean)']:.4f} ± {best['R2 (SD)']:.4f}, "
      f"MSE = {best['MSE (mean)']:.3f})")
print("  Note: MSE in (kJ/mol)².")

In [ ]:
# ── Save results ────────────────────────────────────────────────────────
out = f"CV_results_{DESC_TAG}.xlsx"
with pd.ExcelWriter(out) as w:
    results.to_excel(w, sheet_name='Summary', index=False)
    fold_results.to_excel(w, sheet_name='Per-fold', index=False)
    pd.DataFrame({'Molecule Name': df['Molecule Name'], 'SMILES': df['SMILES'], 'HV (actual)': y,
                  **{f'HV pred ({k})': v for k, v in oof_pred.items()}}).to_excel(w, sheet_name='Out-of-fold predictions', index=False)
print("Saved:", out)
try:
    files.download(out)
except Exception:
    pass